# Dataset Cleaning and Preprocessing for Machine Learning

This notebook prepares the dataset for two classification models:
- **Logistic Regression**
- **Random Forest**

Target variable: **`Recidive`**

The workflow is organized step by step and includes both code cells and markdown explanations.

## 1. Import Libraries

We import the libraries needed for:
- data loading and inspection
- preprocessing
- model training
- performance evaluation
- computational measurements such as time, memory, and energy

In [37]:
from pathlib import Path
from time import perf_counter
import tracemalloc
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

warnings.filterwarnings("ignore")

try:
    from codecarbon import EmissionsTracker
    CODECARBON_AVAILABLE = True
except Exception:
    EmissionsTracker = None
    CODECARBON_AVAILABLE = False

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

## 2. Load the Dataset

The notebook expects the Excel file to be stored in the same folder as this notebook:

- `Data-clean/data manule clean.xlsx`

If the file is not found, the notebook will raise a clear error message.

In [38]:
DATA_PATH = Path("data manule clean.xlsx")
TARGET_COLUMN = "Recidive"
TEST_SIZE = 0.20
RANDOM_STATE = 42

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Place 'data manule clean.xlsx' in the same folder as this notebook."
    )

df = pd.read_excel(DATA_PATH)

print(f"Dataset loaded from: {DATA_PATH.resolve()}")
print(f"Shape: {df.shape}")

Dataset loaded from: D:\Test IA\Data-clean\data manule clean.xlsx
Shape: (20, 32)


## 3. Inspect the Data

In this section, we inspect:
- the first rows
- column names
- data types
- missing values
- duplicate rows

This inspection helps us understand what needs preprocessing before training the models.

In [39]:
display(df.head())

print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values per column:")
display(df.isna().sum())

duplicate_count = df.duplicated().sum()
print(f"\nNumber of duplicate rows: {duplicate_count}")

,numfich,AGE,motifconsult,stress,palpitations,spp,amg,diarrhee,temeblements,agitation,troublehumeur,sommeil,hypersud,thermophobie,faiblessemusc,goitre,classifgoitre,TSH,FT4,AntiTPO,AntiTPOTAUX,AntiTg,TSI,TSItaux,Echographie,Scintigraphie,Therapie,blockandrep,Recidive,duréeATS,chirurgie,IRA
0,1,45,DYSTHYROIDIE,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0.005,51.34,POSITIFS,600.0,POSITIFS,POSITIFS,7.72,goitre,normocaptante,ATS,0.0,1.0,96.0,0,1
1,2,38,DYSTHYROIDIE,1,1,1,1,0,1,0,1,1,0,1,1,0,0,0.001,16.43,POSITIFS,197.1,POSITIFS,POSITIFS,75.24,goitre,hypercaptante,ATS,0.0,0.0,15.0,0,0
2,3,19,DYSTHYROIDIE,1,1,0,0,0,1,0,0,0,1,1,1,1,1A,0.010,98.81,NaN,NaN,NaN,POSITIFS,15.78,goitre,hypercaptante,ATS,0.0,0.0,11.0,1,0
3,4,29,DYSTHYROIDIE,1,1,0,1,1,0,0,1,1,1,1,1,0,0,0.001,22.54,NEGATIFS,0.8,NEGATIFS,NEGATIFS,0.98,volume normal,hypercaptante,ATS,0.0,0.0,18.0,0,0
4,5,34,Signes de compression,1,1,1,1,0,1,1,1,1,1,1,1,0,0,0.050,58.79,NEGATIFS,4.1,NaN,NaN,NaN,volume normal,hypercaptante,ATS,0.0,1.0,18.0,1,0


Columns:
['numfich', 'AGE', 'motifconsult', 'stress', 'palpitations', 'spp', 'amg', 'diarrhee', 'temeblements', 'agitation', 'troublehumeur', 'sommeil', 'hypersud', 'thermophobie', 'faiblessemusc', 'goitre', 'classifgoitre', 'TSH', 'FT4', 'AntiTPO', 'AntiTPOTAUX', 'AntiTg', 'TSI', 'TSItaux', 'Echographie', 'Scintigraphie', 'Therapie', 'blockandrep', 'Recidive', 'duréeATS', 'chirurgie', 'IRA']

Data types:


numfich            int64
AGE                int64
motifconsult      object
stress             int64
palpitations       int64
spp                int64
amg                int64
diarrhee           int64
temeblements       int64
agitation          int64
troublehumeur      int64
sommeil            int64
hypersud           int64
thermophobie       int64
faiblessemusc      int64
goitre             int64
classifgoitre     object
TSH              float64
FT4              float64
AntiTPO           object
AntiTPOTAUX      float64
AntiTg            object
TSI               object
TSItaux          float64
Echographie       object
Scintigraphie     object
Therapie          object
blockandrep      float64
Recidive         float64
duréeATS         float64
chirurgie          int64
IRA                int64
dtype: object


Missing values per column:


numfich          0
AGE              0
motifconsult     0
stress           0
palpitations     0
spp              0
amg              0
diarrhee         0
temeblements     0
agitation        0
troublehumeur    0
sommeil          0
hypersud         0
thermophobie     0
faiblessemusc    0
goitre           0
classifgoitre    0
TSH              0
FT4              2
AntiTPO          4
AntiTPOTAUX      6
AntiTg           6
TSI              3
TSItaux          3
Echographie      2
Scintigraphie    3
Therapie         1
blockandrep      1
Recidive         1
duréeATS         3
chirurgie        0
IRA              0
dtype: int64


Number of duplicate rows: 0


## 4. Remove Irrelevant Variables

Before defining `X` and `y`, we remove columns that should not be used for machine learning.

### What is removed here

- administrative or identifier variables that do not help prediction
- columns that do not provide useful predictive information

### How this is handled

- The irrelevant variable `numfich` is removed before modeling.
- The target column is protected and will never be dropped here.


In [40]:
IRRELEVANT_COLUMNS = [
    "numfich",
]

AUTO_DROP_IDENTIFIER_COLUMNS = False


def suggest_identifier_columns(columns):
    suggestions = []
    known_names = {
        "id", "patient_id", "record_id", "code_patient", "patient_code",
        "num_dossier", "matricule", "identifier"
    }

    for col in columns:
        normalized = col.strip().lower().replace(" ", "_")
        if (
            normalized in known_names
            or normalized.endswith("_id")
            or normalized.startswith("id_")
        ):
            suggestions.append(col)

    return suggestions


identifier_suggestions = suggest_identifier_columns(df.columns)
print("Suggested identifier columns to review:")
print(identifier_suggestions if identifier_suggestions else "No obvious identifier columns detected.")

columns_to_drop = [col for col in IRRELEVANT_COLUMNS if col != TARGET_COLUMN]
missing_drop_columns = [col for col in columns_to_drop if col not in df.columns]
columns_to_drop = [col for col in columns_to_drop if col in df.columns]

if AUTO_DROP_IDENTIFIER_COLUMNS:
    columns_to_drop.extend(identifier_suggestions)

columns_to_drop = sorted(set(columns_to_drop))

if missing_drop_columns:
    print("\nRequested drop columns not found in the dataset:")
    print(missing_drop_columns)

print("\nColumns removed before modeling:")
print(columns_to_drop if columns_to_drop else "No columns removed.")

df_selected = df.drop(columns=columns_to_drop, errors="ignore").copy()

print(f"\nShape before column removal: {df.shape}")
print(f"Shape after column removal: {df_selected.shape}")
display(df_selected.head())


Suggested identifier columns to review:
No obvious identifier columns detected.

Columns removed before modeling:
['numfich']

Shape before column removal: (20, 32)
Shape after column removal: (20, 31)


,AGE,motifconsult,stress,palpitations,spp,amg,diarrhee,temeblements,agitation,troublehumeur,sommeil,hypersud,thermophobie,faiblessemusc,goitre,classifgoitre,TSH,FT4,AntiTPO,AntiTPOTAUX,AntiTg,TSI,TSItaux,Echographie,Scintigraphie,Therapie,blockandrep,Recidive,duréeATS,chirurgie,IRA
0,45,DYSTHYROIDIE,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0.005,51.34,POSITIFS,600.0,POSITIFS,POSITIFS,7.72,goitre,normocaptante,ATS,0.0,1.0,96.0,0,1
1,38,DYSTHYROIDIE,1,1,1,1,0,1,0,1,1,0,1,1,0,0,0.001,16.43,POSITIFS,197.1,POSITIFS,POSITIFS,75.24,goitre,hypercaptante,ATS,0.0,0.0,15.0,0,0
2,19,DYSTHYROIDIE,1,1,0,0,0,1,0,0,0,1,1,1,1,1A,0.010,98.81,NaN,NaN,NaN,POSITIFS,15.78,goitre,hypercaptante,ATS,0.0,0.0,11.0,1,0
3,29,DYSTHYROIDIE,1,1,0,1,1,0,0,1,1,1,1,1,0,0,0.001,22.54,NEGATIFS,0.8,NEGATIFS,NEGATIFS,0.98,volume normal,hypercaptante,ATS,0.0,0.0,18.0,0,0
4,34,Signes de compression,1,1,1,1,0,1,1,1,1,1,1,1,0,0,0.050,58.79,NEGATIFS,4.1,NaN,NaN,NaN,volume normal,hypercaptante,ATS,0.0,1.0,18.0,1,0


## 5. Separate Features `X` and Target `y`

Before training, we separate:
- `X`: all predictor variables
- `y`: the target variable `Recidive`

### How this is handled

Rows with missing values in the target are removed **only for the target definition step**, because supervised machine learning cannot train on samples without a label. The feature columns themselves are not manually cleaned here; their missing values will be handled later inside the preprocessing pipeline.

In [41]:
if TARGET_COLUMN not in df_selected.columns:
    raise KeyError(f"Target column '{TARGET_COLUMN}' was not found in the dataset.")

df_model = df_selected.dropna(subset=[TARGET_COLUMN]).copy()

X = df_model.drop(columns=[TARGET_COLUMN])
y = df_model[TARGET_COLUMN]

print(f"Shape after removing rows with missing target only: {df_model.shape}")
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print("\nTarget distribution:")
display(y.value_counts(dropna=False))

Shape after removing rows with missing target only: (19, 31)
Features shape: (19, 30)
Target shape: (19,)

Target distribution:


Recidive
1.0    10
0.0     9
Name: count, dtype: int64

## 6. Handle Missing Values

### How I did it

- For **numerical features**, missing values are replaced with the **median**.
- For **categorical features**, missing values are replaced with the **most frequent value**.

This is done inside a preprocessing pipeline so the transformations are learned from the training set and then applied consistently to the test set.

## 7. Encode Categorical Variables

### How I did it

Categorical variables are transformed using **One-Hot Encoding**.

- Each category becomes a binary column.
- `handle_unknown='ignore'` is used so that if a category appears in the test set but not in training, the model can still run safely.

## 8. Apply the Required Scaling Strategy

### How I did it

Scaling is applied only to the variables you specified.

- `TSH`, `FT4`, `TSItaux`, `AntiTPOTAUX`, and `duréeATS` are scaled with `RobustScaler`.
- `AGE` is scaled with `StandardScaler`.
- Binary variables are not scaled.
- Other numerical variables, if they exist, are imputed but not scaled.

This keeps the preprocessing aligned with your exact requirement.

In [42]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

robust_scale_candidates = ["TSH", "FT4", "TSItaux", "AntiTPOTAUX", "duréeATS"]
standard_scale_candidates = ["AGE"]

robust_scaled_features = [col for col in robust_scale_candidates if col in X.columns]
standard_scaled_features = [col for col in standard_scale_candidates if col in X.columns]

missing_requested_scaling_columns = [
    col
    for col in robust_scale_candidates + standard_scale_candidates
    if col not in X.columns
]

binary_numeric_features = []
for col in numeric_features:
    if col in robust_scaled_features or col in standard_scaled_features:
        continue

    unique_values = set(pd.Series(X[col]).dropna().unique().tolist())
    if unique_values and unique_values.issubset({0, 1}):
        binary_numeric_features.append(col)

remaining_numeric_features = [
    col for col in numeric_features
    if col not in robust_scaled_features + standard_scaled_features + binary_numeric_features
]

print("Numerical features:")
print(numeric_features)
print("\nCategorical features:")
print(categorical_features)
print("\nRobust-scaled features:")
print(robust_scaled_features)
print("\nStandard-scaled features:")
print(standard_scaled_features)
print("\nBinary numeric features left unscaled:")
print(binary_numeric_features)
print("\nOther numeric features imputed without scaling:")
print(remaining_numeric_features)

if missing_requested_scaling_columns:
    print("\nRequested scaling columns not found in the dataset:")
    print(missing_requested_scaling_columns)

robust_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])

standard_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

remaining_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

transformers = []

if robust_scaled_features:
    transformers.append(("robust_num", robust_numeric_transformer, robust_scaled_features))

if standard_scaled_features:
    transformers.append(("standard_num", standard_numeric_transformer, standard_scaled_features))

if binary_numeric_features:
    transformers.append(("binary_num", binary_numeric_transformer, binary_numeric_features))

if remaining_numeric_features:
    transformers.append(("other_num", remaining_numeric_transformer, remaining_numeric_features))

if categorical_features:
    transformers.append(("cat", categorical_transformer, categorical_features))

preprocessor = ColumnTransformer(transformers=transformers)

Numerical features:
['AGE', 'stress', 'palpitations', 'spp', 'amg', 'diarrhee', 'temeblements', 'agitation', 'troublehumeur', 'sommeil', 'hypersud', 'thermophobie', 'faiblessemusc', 'goitre', 'TSH', 'FT4', 'AntiTPOTAUX', 'TSItaux', 'blockandrep', 'duréeATS', 'chirurgie', 'IRA']

Categorical features:
['motifconsult', 'classifgoitre', 'AntiTPO', 'AntiTg', 'TSI', 'Echographie', 'Scintigraphie', 'Therapie']

Robust-scaled features:
['TSH', 'FT4', 'TSItaux', 'AntiTPOTAUX', 'duréeATS']

Standard-scaled features:
['AGE']

Binary numeric features left unscaled:
['stress', 'palpitations', 'spp', 'amg', 'diarrhee', 'temeblements', 'agitation', 'troublehumeur', 'sommeil', 'hypersud', 'thermophobie', 'faiblessemusc', 'goitre', 'blockandrep', 'chirurgie', 'IRA']

Other numeric features imputed without scaling:
[]


## 9. Split the Data into Training and Testing Sets

### How I did it

- The data is split into **80% training** and **20% testing**.
- `random_state=42` is used for reproducibility.
- `stratify=y` is used to preserve the class distribution in both sets.
- The split is done **before** fitting the imputer, encoder, and scaler to avoid data leakage.

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (15, 30)
X_test shape: (4, 30)
y_train shape: (15,)
y_test shape: (4,)


## 10. Save the Cleaned and Preprocessed Data

### How I saved it

- I fit the preprocessing steps on the **training set only**.
- Then I transformed both the training and testing features.
- I converted the transformed data back into tables with column names.
- Finally, I saved the results to files so they are ready for machine learning.

This avoids data leakage and gives you ready-to-use processed datasets.

Use the saved `train_preprocessed.csv` and `test_preprocessed.csv` files for modeling. The combined file is mainly for inspection or reporting.

For readability, the exported file keeps the original-style variable names whenever possible. Only one-hot encoded categorical variables keep a suffix such as `variable_modalite` so that each generated column remains unique.


In [44]:
def transformed_to_dataframe(transformed_data, feature_names, index):
    if hasattr(transformed_data, "toarray"):
        return pd.DataFrame.sparse.from_spmatrix(
            transformed_data,
            columns=feature_names,
            index=index,
        )
    return pd.DataFrame(transformed_data, columns=feature_names, index=index)


def clean_exported_feature_names(feature_names):
    cleaned_names = []
    prefixes_to_remove = [
        "robust_num__",
        "standard_num__",
        "binary_num__",
        "other_num__",
        "cat__",
    ]

    for name in feature_names:
        clean_name = name
        for prefix in prefixes_to_remove:
            if clean_name.startswith(prefix):
                clean_name = clean_name[len(prefix):]
                break
        cleaned_names.append(clean_name)

    unique_names = []
    counts = {}
    for name in cleaned_names:
        if name not in counts:
            counts[name] = 0
            unique_names.append(name)
        else:
            counts[name] += 1
            unique_names.append(f"{name}_{counts[name]}")

    return unique_names


preprocessing_start_time = perf_counter()

export_preprocessor = clone(preprocessor)
X_train_processed = export_preprocessor.fit_transform(X_train)
X_test_processed = export_preprocessor.transform(X_test)
processed_feature_names = export_preprocessor.get_feature_names_out()
clean_feature_names = clean_exported_feature_names(processed_feature_names)

X_train_processed_df = transformed_to_dataframe(
    X_train_processed,
    clean_feature_names,
    X_train.index,
)
X_test_processed_df = transformed_to_dataframe(
    X_test_processed,
    clean_feature_names,
    X_test.index,
)

train_processed_df = X_train_processed_df.copy()
train_processed_df[TARGET_COLUMN] = y_train.values

test_processed_df = X_test_processed_df.copy()
test_processed_df[TARGET_COLUMN] = y_test.values

full_processed_df = pd.concat([train_processed_df, test_processed_df]).sort_index()

TRAIN_OUTPUT_CSV = Path("train_preprocessed.csv")
TEST_OUTPUT_CSV = Path("test_preprocessed.csv")
FULL_OUTPUT_CSV = Path("full_preprocessed_dataset.csv")

train_processed_df.to_csv(TRAIN_OUTPUT_CSV, index=False)
test_processed_df.to_csv(TEST_OUTPUT_CSV, index=False)
full_processed_df.to_csv(FULL_OUTPUT_CSV, index=False)

preprocessing_end_time = perf_counter()
preprocessing_time_seconds = preprocessing_end_time - preprocessing_start_time
preprocessing_time_summary_df = pd.DataFrame(
    {
        "Step": ["Data preprocessing and export"],
        "Execution Time (s)": [preprocessing_time_seconds],
        "Execution Time (min)": [preprocessing_time_seconds / 60],
    }
)

print(f"Saved training data to: {TRAIN_OUTPUT_CSV.resolve()}")
print(f"Saved testing data to: {TEST_OUTPUT_CSV.resolve()}")
print(f"Saved full processed data to: {FULL_OUTPUT_CSV.resolve()}")
print(f"Preprocessing time: {preprocessing_time_seconds:.6f} seconds")

display(full_processed_df.head())


Saved training data to: D:\Test IA\Data-clean\train_preprocessed.csv
Saved testing data to: D:\Test IA\Data-clean\test_preprocessed.csv
Saved full processed data to: D:\Test IA\Data-clean\full_preprocessed_dataset.csv
Preprocessing time: 0.024446 seconds


,TSH,FT4,TSItaux,AntiTPOTAUX,duréeATS,AGE,stress,palpitations,spp,amg,diarrhee,temeblements,agitation,troublehumeur,sommeil,hypersud,thermophobie,faiblessemusc,goitre,blockandrep,chirurgie,IRA,motifconsult_DYSTHYROIDIE,motifconsult_Signes de compression,classifgoitre_0,classifgoitre_1A,classifgoitre_2,classifgoitre_3,AntiTPO_NEGATIFS,AntiTPO_POSITIFS,AntiTg_NEGATIFS,AntiTg_POSITIFS,TSI_NEGATIFS,TSI_POSITIFS,Echographie_goitre,Echographie_goitre + nodules,Echographie_volume normal,Scintigraphie_hypercaptante,Scintigraphie_nodule chaud,Scintigraphie_normocaptante,Therapie_ATS,Recidive
0,-0.111111,0.524539,-0.323251,5.040236,2.88,0.753189,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
1,-0.200000,-0.474460,6.058601,1.437768,-0.36,0.140128,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,0.000000,1.882959,0.438563,0.000000,-0.52,-1.523894,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,-0.200000,-0.299614,-0.960302,-0.317418,-0.24,-0.648093,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
4,0.888889,0.737731,0.000000,-0.287911,-0.24,-0.210192,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0


## 11. Define Evaluation Metrics and Utility Functions

We compute the following classification metrics:
- Accuracy
- Balanced Accuracy
- Precision
- Recall
- Specificity
- Negative Predictive Value (NPV)
- F1 Score
- Matthews Correlation Coefficient (MCC)

We also measure:
- Execution time
- Peak memory usage
- Energy consumption

In [45]:
def get_binary_labels(y_true):
    labels = pd.Series(y_true).dropna().unique().tolist()
    if len(labels) != 2:
        raise ValueError(
            "This notebook expects a binary target for Specificity and NPV calculation. "
            f"Found classes: {labels}"
        )
    labels = sorted(labels)
    negative_label = labels[0]
    positive_label = labels[1]
    return negative_label, positive_label


def compute_metrics(y_true, y_pred):
    negative_label, positive_label = get_binary_labels(y_true)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[negative_label, positive_label],
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
        "Specificity": specificity,
        "NPV": npv,
        "F1 Score": f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }


def evaluate_pipeline(model_name, pipeline, X_train, X_test, y_train, y_test):
    result = {
        "Model": model_name,
        "Accuracy": np.nan,
        "Balanced Accuracy": np.nan,
        "Precision": np.nan,
        "Recall": np.nan,
        "Specificity": np.nan,
        "NPV": np.nan,
        "F1 Score": np.nan,
        "MCC": np.nan,
        "Execution Time (s)": np.nan,
        "Peak Memory (MB)": np.nan,
        "Energy Consumption (kg CO2eq)": np.nan,
        "Status": "Success",
        "Error": None,
    }

    tracker = None
    start_time = perf_counter()
    tracemalloc.start()

    try:
        if CODECARBON_AVAILABLE:
            tracker = EmissionsTracker(save_to_file=False)
            tracker.start()

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        result.update(compute_metrics(y_test, y_pred))

    except Exception as exc:
        result["Status"] = "Failed"
        result["Error"] = f"{type(exc).__name__}: {exc}"

    finally:
        _, peak_memory = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        result["Execution Time (s)"] = perf_counter() - start_time
        result["Peak Memory (MB)"] = peak_memory / (1024 ** 2)

        if tracker is not None:
            try:
                result["Energy Consumption (kg CO2eq)"] = tracker.stop()
            except Exception as exc:
                if result["Error"] is None:
                    result["Error"] = f"Energy tracking failed: {type(exc).__name__}: {exc}"
        else:
            result["Energy Consumption (kg CO2eq)"] = np.nan

    return result

## 12. Train Logistic Regression

The preprocessing steps are combined with the classifier inside a single pipeline.
This ensures the same training-only transformations are applied correctly to the test set.

In [46]:
logistic_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

logistic_result = evaluate_pipeline(
    "Logistic Regression",
    logistic_pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
)

pd.DataFrame([logistic_result])

[codecarbon WARNING @ 16:35:41] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 16:35:43] [setup] RAM Tracking...
[codecarbon INFO @ 16:35:43] [setup] CPU Tracking...
[codecarbon WARNING @ 16:35:43] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Windows OS detected: Please install Intel Power Gadget to measure CPU

[codecarbon INFO @ 16:35:43] CPU Model on constant consumption mode: AMD Ryzen 7 5700X 8-Core Processor
[codecarbon WARNING @ 16:35:43] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon INFO @ 16:35:43] [setup] GPU Tracking...
[codecarbon INFO @ 16:35:43] Tracking Nvidia GPUs via PyNVML
[codecarbon INFO @ 16:35:43] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: cpu_load
                GPU Tracking Method: pynvml
            
[codecarbon INFO @ 16:35:43] >>> Tracker's metadata:
[codeca

,Model,Accuracy,Balanced Accuracy,Precision,Recall,Specificity,NPV,F1 Score,MCC,Execution Time (s),Peak Memory (MB),Energy Consumption (kg CO2eq),Status,Error
0,Logistic Regression,0.75,0.75,1.0,0.5,1.0,0.666667,0.666667,0.57735,2.503042,0.15675,4.699762e-08,Success,None


## 13. Train Random Forest

Random Forest is trained using the same preprocessing pipeline so the comparison is fair and consistent.

In [47]:
random_forest_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200)),
])

random_forest_result = evaluate_pipeline(
    "Random Forest",
    random_forest_pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
)

pd.DataFrame([random_forest_result])

[codecarbon WARNING @ 16:35:45] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 16:35:46] [setup] RAM Tracking...
[codecarbon INFO @ 16:35:46] [setup] CPU Tracking...
[codecarbon WARNING @ 16:35:46] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Windows OS detected: Please install Intel Power Gadget to measure CPU

[codecarbon INFO @ 16:35:46] CPU Model on constant consumption mode: AMD Ryzen 7 5700X 8-Core Processor
[codecarbon WARNING @ 16:35:46] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon INFO @ 16:35:46] [setup] GPU Tracking...
[codecarbon INFO @ 16:35:46] Tracking Nvidia GPUs via PyNVML
[codecarbon INFO @ 16:35:46] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: cpu_load
                GPU Tracking Method: pynvml
            
[codecarbon INFO @ 16:35:46] >>> Tracker's metadata:
[codeca

,Model,Accuracy,Balanced Accuracy,Precision,Recall,Specificity,NPV,F1 Score,MCC,Execution Time (s),Peak Memory (MB),Energy Consumption (kg CO2eq),Status,Error
0,Random Forest,0.75,0.75,1.0,0.5,1.0,0.666667,0.666667,0.57735,4.110381,0.369874,1.353166e-07,Success,None


## 14. Compare the Two Models

This final table compares predictive performance and computational cost for both models.

In [48]:
results_df = pd.DataFrame([logistic_result, random_forest_result])

display_df = results_df.copy()
numeric_columns = display_df.select_dtypes(include=[np.number]).columns
display_df[numeric_columns] = display_df[numeric_columns].round(6)

display(display_df)

,Model,Accuracy,Balanced Accuracy,Precision,Recall,Specificity,NPV,F1 Score,MCC,Execution Time (s),Peak Memory (MB),Energy Consumption (kg CO2eq),Status,Error
0,Logistic Regression,0.75,0.75,1.0,0.5,1.0,0.666667,0.666667,0.57735,2.503042,0.156750,0.0,Success,None
1,Random Forest,0.75,0.75,1.0,0.5,1.0,0.666667,0.666667,0.57735,4.110381,0.369874,0.0,Success,None


## Final Notes

- This notebook is ready for a cleaned dataset used in supervised machine learning.
- The irrelevant variable `numfich` is removed before modeling.
- Missing feature values are handled inside the preprocessing pipeline.
- The train/test split happens before fitting preprocessing steps, which helps prevent data leakage.
- Categorical variables are encoded with One-Hot Encoding.
- `TSH`, `FT4`, `TSItaux`, `AntiTPOTAUX`, and `duréeATS` are scaled with `RobustScaler`.
- `AGE` is scaled with `StandardScaler`.
- Binary numeric variables are left unscaled.
- If `codecarbon` is installed, energy consumption will be measured. Otherwise, the energy column will remain unavailable.
- The preprocessing time is displayed at the end of the notebook.


## Preprocessing Time Summary

This final section reports the total execution time of the preprocessing block, including transformation and export of the cleaned datasets.


In [49]:
preprocessing_time_display = preprocessing_time_summary_df.copy()
preprocessing_time_display[["Execution Time (s)", "Execution Time (min)"]] = preprocessing_time_display[["Execution Time (s)", "Execution Time (min)"]].round(6)

display(preprocessing_time_display)


,Step,Execution Time (s),Execution Time (min)
0,Data preprocessing and export,0.024446,0.000407
